In [1]:
%uv pip install pandas numpy scanpy scrublet

Using Python 3.12.6 environment at: /usr/local
Resolved 50 packages in 213ms
⠙ Preparing packages... (0/17)
⠙ Preparing packages... (0/17)
⠙ Preparing packages... (0/17)
legacy-api-wrap ------------------------------ 9.94 KiB/9.94 KiB
⠙ Preparing packages... (0/17)
legacy-api-wrap ------------------------------ 9.94 KiB/9.94 KiB
session-info2 ------------------------------     0 B/17.28 KiB
⠙ Preparing packages... (0/17)
legacy-api-wrap ------------------------------ 9.94 KiB/9.94 KiB
session-info2 ------------------------------ 14.80 KiB/17.28 KiB
⠙ Preparing packages... (0/17)
legacy-api-wrap ------------------------------ 9.94 KiB/9.94 KiB
session-info2 ------------------------------ 14.80 KiB/17.28 KiB
donfig     ------------------------------     0 B/21.09 KiB
⠙ Preparing packages... (0/17)
legacy-api-wrap ------------------------------ 9.94 KiB/9.94 KiB
session-info2 ------------------------------ 14.80 KiB/17.28 KiB
donfig     ------------------------------ 14.83 KiB/21.09 KiB
⠙

In [2]:
import sys
sys.path.insert(0, '/home/ubuntu/.local/lib/python3.12/site-packages')

In [3]:
!curl -L -o dropbox_folder_metadata.zip "https://www.dropbox.com/scl/fo/0bskm59i6gmlfzwqc1hmo/ABoCoggFt3pBWAA904tMJCQ?rlkey=bbv59xhqd7b7bnz9cs2fqkn6e&dl=1"

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100    17  100    17    0     0     21      0 --:--:-- --:--:-- --:--:--    21
100 7581k  100 7581k    0     0  3070k      0  0:00:02  0:00:02 --:--:-- 5210k


In [4]:
!curl -L -o dropbox_folder_data.zip "https://www.dropbox.com/scl/fo/e0ndj373kk6v890hauvch/AEmF4fOT3Lpo231O_EgP5f0?rlkey=xz9vauv6s9pg1yl57pwteatv8&dl=1"

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100    17  100    17    0     0     23      0 --:--:-- --:--:-- --:--:--    23
100 1656M  100 1656M    0     0  19.3M      0  0:01:25  0:01:25 --:--:-- 19.7M


In [5]:
!unzip "*.zip"  

Archive:  dropbox_folder_metadata.zip
mapname:  conversion of  failed
 extracting: Meta-data_Bi2021_Kidney.tar.gz  
 extracting: Meta-data_Zhang2021_Kidney.tar.gz  
 extracting: Meta-data_Young2018_Kidney.tar.gz  
 extracting: Meta-data_Krishna2021_Kidney.tar.gz  
 extracting: Meta-data_Obradovic2021_Kidney.tar.gz  

Archive:  dropbox_folder_data.zip
mapname:  conversion of  failed
 extracting: Data_Bi2021_Kidney.tar.gz  
 extracting: Data_Zhang2021_Kidney.tar.gz  
 extracting: Data_Young2018_Kidney.tar.gz  
 extracting: Data_Krishna2021_Kidney.tar.gz  
 extracting: Data_Obradovic2021_Kidney.tar.gz  

2 archives had fatal errors.


In [6]:
!for f in *.tar.gz; do tar -xzvf "$f"; done

Data_Bi2021_Kidney/
Data_Bi2021_Kidney/Samples.csv
Data_Bi2021_Kidney/Exp_data_UMIcounts_normalized.mtx
Data_Bi2021_Kidney/Cells.csv
Data_Bi2021_Kidney/Genes.txt
Data_Krishna2021_Kidney/
Data_Krishna2021_Kidney/Samples.csv
Data_Krishna2021_Kidney/Cells.csv
Data_Krishna2021_Kidney/Genes.txt
Data_Krishna2021_Kidney/Exp_data_UMIcounts.mtx
Data_Obradovic2021_Kidney/
Data_Obradovic2021_Kidney/Samples.csv
Data_Obradovic2021_Kidney/Cells.csv
Data_Obradovic2021_Kidney/Genes.txt
Data_Obradovic2021_Kidney/Exp_data_UMIcounts.mtx
Data_Young2018_Kidney/
Data_Young2018_Kidney/Samples.csv
Data_Young2018_Kidney/Cells.csv
Data_Young2018_Kidney/Genes.txt
Data_Young2018_Kidney/Exp_data_UMIcounts.mtx
Data_Zhang2021_Kidney/
Data_Zhang2021_Kidney/Samples.csv
Data_Zhang2021_Kidney/Cells.csv
Data_Zhang2021_Kidney/Genes.txt
Data_Zhang2021_Kidney/Exp_data_UMIcounts.mtx
Meta-data_Bi2021_Kidney/
Meta-data_Bi2021_Kidney/Samples.csv
Meta-data_Bi2021_Kidney/Cells.csv
Meta-data_Krishna2021_Kidney/
Meta-data_Krishna20

In [8]:
import pandas as pd
import numpy as np 
import scanpy as sc 
import os
import scrublet as scr 
import glob
import gc

In [9]:
import anndata as ad
ad.settings.allow_write_nullable_strings = True

In [10]:
study_files = {
    "Bi2021": {
        "matrix": "Data_Bi2021_Kidney/Exp_data_UMIcounts_normalized.mtx",
        "genes":  "Data_Bi2021_Kidney/Genes.txt",
        "cells":  "Data_Bi2021_Kidney/Cells.csv",
        "samples": "Data_Bi2021_Kidney/Samples.csv",
        "units": "UMI"
    },
    "Krishna2021": {
        "matrix": "Data_Krishna2021_Kidney/Exp_data_UMIcounts.mtx",
        "genes":  "Data_Krishna2021_Kidney/Genes.txt",
        "cells":  "Data_Krishna2021_Kidney/Cells.csv",
        "samples": "Data_Krishna2021_Kidney/Samples.csv",
        "units": "UMI"
    },
    "Obradovic2021": {
        "matrix": "Data_Obradovic2021_Kidney/Exp_data_UMIcounts.mtx",
        "genes":  "Data_Obradovic2021_Kidney/Genes.txt",
        "cells":  "Data_Obradovic2021_Kidney/Cells.csv",
        "samples": "Data_Obradovic2021_Kidney/Samples.csv",
        "units": "UMI"
    },
    "Young2018": {
        "matrix": "Data_Young2018_Kidney/Exp_data_UMIcounts.mtx",
        "genes":  "Data_Young2018_Kidney/Genes.txt",
        "cells":  "Data_Young2018_Kidney/Cells.csv",
        "samples": "Data_Young2018_Kidney/Samples.csv",
        "units": "UMI"
    },
    "Zhang2021": {
        "matrix": "Data_Zhang2021_Kidney/Exp_data_UMIcounts.mtx",
        "genes":  "Data_Zhang2021_Kidney/Genes.txt",
        "cells":  "Data_Zhang2021_Kidney/Cells.csv",
        "samples": "Data_Zhang2021_Kidney/Samples.csv",
        "units": "UMI"
    }
}

In [11]:
study_name = 'Bi2021'
paths = study_files[study_name]

# 1. Load matrix and transpose
adata = sc.read_mtx(paths['matrix']).T

# 2. Gene names
genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()   # added for safety

# 3. Cell metadata – fix barcode type before assignment
cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# 4. Drop NaN / suspicious metadata
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# 5. Add cancer_type column (if missing)
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        samples_unique = samples[~samples.index.duplicated(keep='first')]
        sample_to_cancer = samples_unique['cancer_type'].to_dict()
        adata.obs['cancer_type'] = adata.obs['sample'].map(sample_to_cancer)
    else:
        adata.obs['cancer_type'] = 'Kidney'   # fallback
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# 6. Gene‑count filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# 7. Doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# 8. Mitochondrial filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# 9. Normalisation
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:   # UMI
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

Loaded: 34,326 cells × 32,718 genes
   cell_type: 328 NaN
   cell_type: 328 suspicious
Flagged 328 cells
After metadata drop: 33,998 cells
   Added cancer_type column. Unique values: ['Clear Cell Renal Cell Carcinoma' 'Papillary Renal Cell Carcinoma']
After gene filter: 33,899 cells
Preprocessing...


/usr/local/lib/python3.12/site-packages/scrublet/helper_functions.py:252: RuntimeWarning: invalid value encountered in sqrt
  CV_input = np.sqrt(b);


Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.52
Detected doublet rate = 0.3%
Estimated detectable doublet fraction = 27.8%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 0.9%
Elapsed time: 33.5 seconds
After doublet removal: 33,811 cells
After MT filter: 33,810 cells
Normalised (UMI) – max value 10.40


In [12]:
study_name = 'Krishna2021'
paths = study_files[study_name]

# 1. Load matrix and transpose
adata = sc.read_mtx(paths['matrix']).T

# 2. Gene names
genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()   # added for safety

# 3. Cell metadata – fix barcode type before assignment
cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# 4. Drop NaN / suspicious metadata
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# 5. Add cancer_type column (if missing)
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        samples_unique = samples[~samples.index.duplicated(keep='first')]
        sample_to_cancer = samples_unique['cancer_type'].to_dict()
        adata.obs['cancer_type'] = adata.obs['sample'].map(sample_to_cancer)
    else:
        adata.obs['cancer_type'] = 'Kidney'   # fallback
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# 6. Gene‑count filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# 7. Doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# 8. Mitochondrial filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# 9. Normalisation
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:   # UMI
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

Loaded: 167,283 cells × 15,588 genes
   cell_type: 13782 NaN
   cell_type: 13782 suspicious
Flagged 13782 cells
After metadata drop: 153,501 cells
   Added cancer_type column. Unique values: ['Clear Cell Renal Cell Carcinoma']
After gene filter: 153,386 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.67
Detected doublet rate = 0.0%
Estimated detectable doublet fraction = 13.8%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 0.3%
Elapsed time: 293.1 seconds
After doublet removal: 153,328 cells
After MT filter: 153,328 cells
Normalised (UMI) – max value 13.52


In [13]:
study_name = 'Obradovic2021'
paths = study_files[study_name]

# 1. Load matrix and transpose
adata = sc.read_mtx(paths['matrix']).T

# 2. Gene names
genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()   # added for safety

# 3. Cell metadata – fix barcode type before assignment
cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# 4. Drop NaN / suspicious metadata
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# 5. Add cancer_type column (if missing)
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        samples_unique = samples[~samples.index.duplicated(keep='first')]
        sample_to_cancer = samples_unique['cancer_type'].to_dict()
        adata.obs['cancer_type'] = adata.obs['sample'].map(sample_to_cancer)
    else:
        adata.obs['cancer_type'] = 'Kidney'   # fallback
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# 6. Gene‑count filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# 7. Doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# 8. Mitochondrial filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# 9. Normalisation
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:   # UMI
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

Loaded: 19,781 cells × 19,234 genes
   cell_type: 415 NaN
   cell_type: 415 suspicious
Flagged 415 cells
After metadata drop: 19,366 cells
   Added cancer_type column. Unique values: ['Clear Cell Renal Cell Carcinoma']
After gene filter: 19,366 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.69
Detected doublet rate = 0.0%
Estimated detectable doublet fraction = 8.6%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 0.2%
Elapsed time: 15.1 seconds
After doublet removal: 19,363 cells
After MT filter: 19,363 cells
Normalised (UMI) – max value 13.20


In [14]:
study_name = 'Young2018'
paths = study_files[study_name]

# 1. Load matrix and transpose
adata = sc.read_mtx(paths['matrix']).T

# 2. Gene names
genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()   # added for safety

# 3. Cell metadata – fix barcode type before assignment
cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# 4. Drop NaN / suspicious metadata
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# 5. Add cancer_type column (if missing)
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        samples_unique = samples[~samples.index.duplicated(keep='first')]
        sample_to_cancer = samples_unique['cancer_type'].to_dict()
        adata.obs['cancer_type'] = adata.obs['sample'].map(sample_to_cancer)
    else:
        adata.obs['cancer_type'] = 'Kidney'   # fallback
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# 6. Gene‑count filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# 7. Doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# 8. Mitochondrial filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# 9. Normalisation
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:   # UMI
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

Loaded: 125,139 cells × 33,694 genes
   cell_type: 95415 NaN
   cell_type: 95415 suspicious
Flagged 95415 cells
After metadata drop: 29,724 cells
   Added cancer_type column. Unique values: ['Wilms Tumor' 'Clear Cell Renal Cell Carcinoma' nan
 'Papillary Renal Cell Carcinoma' 'Normal']
After gene filter: 29,445 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.60
Detected doublet rate = 0.0%
Estimated detectable doublet fraction = 8.3%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 0.3%
Elapsed time: 28.1 seconds
After doublet removal: 29,437 cells
After MT filter: 28,486 cells
Normalised (UMI) – max value 13.47


In [15]:
study_name = 'Zhang2021'
paths = study_files[study_name]

# 1. Load matrix and transpose
adata = sc.read_mtx(paths['matrix']).T

# 2. Gene names
genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()   # added for safety

# 3. Cell metadata – fix barcode type before assignment
cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# 4. Drop NaN / suspicious metadata
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# 5. Add cancer_type column (if missing)
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        samples_unique = samples[~samples.index.duplicated(keep='first')]
        sample_to_cancer = samples_unique['cancer_type'].to_dict()
        adata.obs['cancer_type'] = adata.obs['sample'].map(sample_to_cancer)
    else:
        adata.obs['cancer_type'] = 'Kidney'   # fallback
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# 6. Gene‑count filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# 7. Doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# 8. Mitochondrial filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# 9. Normalisation
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:   # UMI
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

Loaded: 29,474 cells × 33,694 genes
   cell_type: 345 NaN
   cell_type: 345 suspicious
Flagged 345 cells
After metadata drop: 29,129 cells
   Added cancer_type column. Unique values: ['Clear Cell Renal Cell Carcinoma' nan 'Chromophobe Renal Cell Carcinoma']
After gene filter: 28,863 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.55
Detected doublet rate = 0.0%
Estimated detectable doublet fraction = 23.5%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 0.0%
Elapsed time: 31.3 seconds
After doublet removal: 28,860 cells
After MT filter: 24,157 cells
Normalised (UMI) – max value 13.34


In [16]:
INPUT_PATTERN = "*.h5ad"   

files = sorted(glob.glob(INPUT_PATTERN))

for f in files:
    adata = sc.read_h5ad(f)
    study_name = adata.obs['study'].iloc[0] if 'study' in adata.obs.columns else f
    print(f"{study_name}  –  {adata.n_obs:,} cells, {adata.n_vars:,} genes")
    
    for col in sorted(adata.obs.columns):
        if pd.api.types.is_numeric_dtype(adata.obs[col]):
            continue
        
        vals = adata.obs[col].dropna().unique()
        n_unique = len(vals)
        
        if n_unique <= 25:
            val_str = ', '.join(str(v) for v in sorted(vals))
        else:
            val_str = f"{n_unique} unique values – first 15: " + ', '.join(str(v) for v in sorted(vals)[:15])
        
        print(f" {col}: {val_str}")
    
    print()

Bi2021  –  33,810 cells, 32,718 genes
 cancer_type: Clear Cell Renal Cell Carcinoma, Papillary Renal Cell Carcinoma
 cell_cycle_phase: G1/S, G2/M, Intermediate, Not cycling, nan
 cell_lineage: Lymphoid, Myeloid, Normal Tissue, Putative Tumor
 cell_subtype: 32 unique values – first 15: 41BB-Hi CD8+ T cell, 41BB-Lo CD8+ T cell, B cell, CD16+ Monocyte, CD16- Monocyte, CD1C+ DC, CLEC9A+ DC, CXCL10-Hi TAM, Cycling CD8+ T cell, Cycling TAM, Cycling Tumor, Effector T-Helper, Endothelial, FGFBP2+ NK, FGFBP2- NK
 cell_type: B_cell, Dendritic, Endothelial, Fibroblast, Macrophage, Malignant, Mast, Monocyte, Myeloid, NK_cell, Plasma, T_cell
 disease: clear cell renal carcinoma, papillary renal cell carcinoma
 gender: female, male
 mp_assignment: 42 unique values – first 15: CD4 - Cell Cycle, CD4 - Cytotoxic, CD4 - Dysfunction, CD4 - Glycolysis/MYC, CD4 - Naive1, CD4 - Naive2, CD4 - Treg, CD8 - Cell Cycle, CD8 - Cytotoxic, CD8 - Glycolysis/MYC, CD8 - Memory/Naive1, CD8 - Naive2, CD8 - Unassigned2, 

In [18]:
import scanpy as sc
import pandas as pd
import numpy as np
import glob
import os
import gc
import anndata as ad
import modal 

VOLUME_NAME = "output"  
vol = modal.Volume.from_name(VOLUME_NAME)

ad.settings.allow_write_nullable_strings = True

INPUT_PATTERN = "*.h5ad"
OUTPUT_FILE = "/mnt/output/kidney.h5ad"  

VALID_PREFIXES = [
    'Bi2021', 'Krishna2021', 'Obradovic2021',
    'Young2018', 'Zhang2021'
]

all_files = sorted(glob.glob(INPUT_PATTERN))
study_files = [f for f in all_files if any(f.startswith(p) for p in VALID_PREFIXES)]
print(f"Found {len(study_files)} study files (expected 5)\n")

def label_malignant(adata, study_name):
    if 'cell_type' in adata.obs.columns:
        ct = adata.obs['cell_type'].astype(str).str.lower().str.strip()
        if ct.str.contains('malignant', na=False).any():
            return ct.str.contains('malignant', na=False), "cell_type='Malignant'"
    if 'is_tumor' in adata.obs.columns:
        it = adata.obs['is_tumor'].astype(str).str.lower().str.strip()
        if it.str.contains('yes', na=False).any():
            return it.str.contains('yes', na=False), "is_tumor='YES'"
    if 'source' in adata.obs.columns:
        src = adata.obs['source'].astype(str).str.lower().str.strip()
        if src.str.contains('tumor|tumour', na=False).any():
            return src.str.contains('tumor|tumour', na=False), "source='Tumor'"
    return pd.Series(False, index=adata.obs.index), "all non-malignant"

temp_files = []
total_malig = 0
total_non = 0

for f in study_files:
    adata = sc.read_h5ad(f)
    study_name = adata.obs['study'].iloc[0]
    print(f"Processing {study_name} ({adata.n_obs} cells) …")

    if 'cancer_type' not in adata.obs.columns or adata.obs['cancer_type'].isna().all():
        adata.obs['cancer_type'] = 'Kidney Cancer'
    else:
        adata.obs['cancer_type'] = adata.obs['cancer_type'].astype(str).fillna('Kidney Cancer')

    malignant_mask, strategy = label_malignant(adata, study_name)
    adata.obs['is_malignant'] = malignant_mask.astype(int)

    malig_count = malignant_mask.sum()
    non_count = (~malignant_mask).sum()
    total_malig += malig_count
    total_non += non_count
    print(f"   malignant: {malig_count:>6,}   non‑malignant: {non_count:>7,}   ({strategy})")

    essential = ['sample', 'cell_type', 'cancer_type', 'study', 'is_malignant']
    keep = [c for c in essential if c in adata.obs.columns]
    adata.obs = adata.obs[keep]

    for col in adata.obs.columns:
        try:
            adata.obs[col] = adata.obs[col].astype(str)
        except Exception:
            pass
    adata.obs_names = adata.obs_names.astype(str)

    temp_file = f"__temp_{study_name}.h5ad"
    adata.write_h5ad(temp_file)
    temp_files.append(temp_file)

print(f"\n{'='*60}")
print(f"TOTAL: {total_malig+total_non:>7,} cells  →  malignant: {total_malig:>6,}   non‑malignant: {total_non:>7,}")

print(f"\nMerging {len(temp_files)} studies incrementally …")
adata_all = sc.read_h5ad(temp_files[0])
print(f"   Starting with: {adata_all.n_obs:,} cells × {adata_all.n_vars:,} genes")

for i, tf in enumerate(temp_files[1:], 1):
    print(f"   Adding study {i} …")
    ad = sc.read_h5ad(tf)
    adata_all = adata_all.concatenate(ad, batch_key='study', join='inner', index_unique='-')
    del ad
    gc.collect()
    print(f"      Combined now: {adata_all.n_obs:,} cells × {adata_all.n_vars:,} genes")

print(f"\nFinal merged: {adata_all.n_obs:,} cells × {adata_all.n_vars:,} genes")

study_names_original = [sc.read_h5ad(tf).obs['study'].iloc[0] for tf in temp_files]
study_map = {str(i): name for i, name in enumerate(study_names_original)}
adata_all.obs['study'] = adata_all.obs['study'].astype(str).map(study_map)
print(f"Study names restored: {sorted(adata_all.obs['study'].unique())}")

print("\nCancer types:", sorted(adata_all.obs['cancer_type'].astype(str).unique()))
print("Malignant distribution:")
print(adata_all.obs['is_malignant'].value_counts())

for col in adata_all.obs.columns:
    n_nan = adata_all.obs[col].isna().sum()
    if n_nan > 0:
        print(f"{col}: {n_nan} NaN – dropping")
        adata_all = adata_all[~adata_all.obs[col].isna(), :].copy()

for col in adata_all.obs.columns:
    try:
        adata_all.obs[col] = adata_all.obs[col].astype(str)
    except Exception:
        pass
adata_all.obs_names = adata_all.obs_names.astype(str)

os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)
adata_all.write_h5ad(OUTPUT_FILE)

print(f"Saved to volume: {OUTPUT_FILE}")

for tf in temp_files:
    os.remove(tf)
print("Temporary files cleaned up.")

Found 5 study files (expected 5)

Processing Bi2021 (33810 cells) …
   malignant:  7,921   non‑malignant:  25,889   (cell_type='Malignant')
Processing Krishna2021 (153328 cells) …
   malignant:  5,689   non‑malignant: 147,639   (cell_type='Malignant')
Processing Obradovic2021 (19363 cells) …
   malignant:  9,368   non‑malignant:   9,995   (cell_type='Malignant')
Processing Young2018 (28486 cells) …
   malignant:  1,881   non‑malignant:  26,605   (cell_type='Malignant')
Processing Zhang2021 (24157 cells) …
   malignant:  8,823   non‑malignant:  15,334   (cell_type='Malignant')

TOTAL: 259,144 cells  →  malignant: 33,682   non‑malignant: 225,462

Merging 5 studies incrementally …
   Starting with: 33,810 cells × 32,718 genes
   Adding study 1 …


/tmp/ipykernel_176/1318961009.py:90: FutureWarning: The method concatenate is deprecated and will be removed in the future. Use anndata.concat instead of AnnData.concatenate. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  adata_all = adata_all.concatenate(ad, batch_key='study', join='inner', index_unique='-')


      Combined now: 187,138 cells × 15,069 genes
   Adding study 2 …


/tmp/ipykernel_176/1318961009.py:90: FutureWarning: The method concatenate is deprecated and will be removed in the future. Use anndata.concat instead of AnnData.concatenate. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  adata_all = adata_all.concatenate(ad, batch_key='study', join='inner', index_unique='-')


      Combined now: 206,501 cells × 14,423 genes
   Adding study 3 …


/tmp/ipykernel_176/1318961009.py:90: FutureWarning: The method concatenate is deprecated and will be removed in the future. Use anndata.concat instead of AnnData.concatenate. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  adata_all = adata_all.concatenate(ad, batch_key='study', join='inner', index_unique='-')


      Combined now: 234,987 cells × 14,022 genes
   Adding study 4 …


/tmp/ipykernel_176/1318961009.py:90: FutureWarning: The method concatenate is deprecated and will be removed in the future. Use anndata.concat instead of AnnData.concatenate. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  adata_all = adata_all.concatenate(ad, batch_key='study', join='inner', index_unique='-')


      Combined now: 259,144 cells × 14,022 genes

Final merged: 259,144 cells × 14,022 genes
Study names restored: ['Bi2021', 'Krishna2021']

Cancer types: ['Chromophobe Renal Cell Carcinoma', 'Clear Cell Renal Cell Carcinoma', 'Normal', 'Papillary Renal Cell Carcinoma', 'Wilms Tumor', 'nan']
Malignant distribution:
is_malignant
0    225462
1     33682
Name: count, dtype: int64
Saved to volume: /mnt/output/kidney.h5ad
Temporary files cleaned up.


In [ ]:
# auth: model token new

In [ ]:
# modal volume get output /output/kidney.h5ad ./kidney.h5ad